# 职业分类 - 一位码（0-10）
基于阿里通义千问API，将识别结果中的职业描述匹配到《中国职业大典》一位码分类。

输入：`结果/1.识别结果/` 下的人头计数表和独立职业表
输出：`结果/2.分类结果/一位码/` 下的已分类表格

In [ ]:
import pandas as pd
import json
import os
import time
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# 加载 .env（职业分类阶段的API密钥）
load_dotenv()

# API 配置
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen-plus-2025-12-01")

if not API_KEY:
    raise ValueError("请在 2.职业分类/.env 文件中设置 API_KEY")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# 项目路径
PROJECT_ROOT = Path("..").resolve()
INPUT_DIR = PROJECT_ROOT / "结果" / "1.识别结果"
OUTPUT_DIR = PROJECT_ROOT / "结果" / "2.分类结果" / "一位码"
CLASS_FILE = Path("职业分类.xlsx")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"输入目录: {INPUT_DIR}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"分类表: {CLASS_FILE}")

In [ ]:
# 加载分类表
df_class = pd.read_excel(CLASS_FILE)

# 动态补充第10类
if 10 not in df_class['代码'].values:
    new_row = pd.DataFrame([{
        '代码': 10, '分类名称': '非从业人员-其他',
        '分类内容': '无法归入学生、儿童或家庭角色的其他非从业人员，或身份不明者。'
    }])
    df_class = pd.concat([df_class, new_row], ignore_index=True)

# 构建分类参考字符串
class_info_full = ""
for _, row in df_class.iterrows():
    content = str(row.get('分类内容', ''))[:50]
    class_info_full += f"代码[{row['代码']}] {row['分类名称']}: {content}\n"

# 限制分类（仅0/9/10）
restricted_codes = [0, 9, 10]
class_info_restricted = ""
for _, row in df_class[df_class['代码'].isin(restricted_codes)].iterrows():
    content = str(row.get('分类内容', ''))[:50]
    class_info_restricted += f"代码[{row['代码']}] {row['分类名称']}: {content}\n"

code_to_name = df_class.set_index('代码')['分类名称'].to_dict()

print("分类表加载完成:")
print(class_info_full)

In [ ]:
def classify_batch(descriptions: list, class_info_str: str, restricted: bool = False) -> dict:
    """调用大模型对一批描述进行职业分类"""
    if restricted:
        constraint = """【重要限制】这组数据通过视觉无法确认职业，属于非从业人员场景。
请仅从以下三个类别中选择，严禁归入其他职业类：
- 0 (学生/儿童)
- 9 (家庭角色，如母亲、父亲、祖辈)
- 10 (其他非从业人员，或无法判断的非职业身份)"""
    else:
        constraint = "请根据描述归入最合适的类别（0-10类）。如果描述的是学生、儿童或家庭角色，请优先归入0或9类。"

    prompt = f"""任务：请将以下人物描述归类到《职业分类大典》的对应类别中。

参考分类标准：
{class_info_str}

{constraint}

待分类的描述列表：
{json.dumps(descriptions, ensure_ascii=False)}

要求：
1. 严格输出JSON格式，Key为原描述，Value为对应的"代码"（数字）。
2. 不要输出任何解释性文字。

输出示例：
{{"正在讲课的老师": 2, "背书包的小学生": 0, "做饭的母亲": 9}}"""

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"  API调用出错: {e}")
        return {}

def process_file(file_path: Path, output_path: Path):
    """处理单个识别结果文件，进行一位码分类"""
    print(f"\n处理: {file_path.name}")
    df = pd.read_excel(file_path)

    if df.empty:
        print("  空文件，跳过")
        return

    mapping_dict = {}
    BATCH_SIZE = 50

    # 轨道A：profession != "未知" → 用profession匹配全量分类表
    known_mask = df['profession'] != '未知'
    descs_known = df.loc[known_mask, 'profession'].dropna().unique().tolist()
    if descs_known:
        print(f"  轨道A (已知职业): {len(descs_known)} 个描述")
        for i in range(0, len(descs_known), BATCH_SIZE):
            batch = descs_known[i:i+BATCH_SIZE]
            res = classify_batch(batch, class_info_full, restricted=False)
            mapping_dict.update(res)
            time.sleep(0.3)

    # 轨道B：profession == "未知" → 用identifier匹配限制分类表(0/9/10)
    unknown_mask = df['profession'] == '未知'
    descs_unknown = df.loc[unknown_mask, 'identifier'].dropna().unique().tolist()
    if descs_unknown:
        print(f"  轨道B (未知职业): {len(descs_unknown)} 个描述")
        for i in range(0, len(descs_unknown), BATCH_SIZE):
            batch = descs_unknown[i:i+BATCH_SIZE]
            res = classify_batch(batch, class_info_restricted, restricted=True)
            mapping_dict.update(res)
            time.sleep(0.3)

    # 回填
    def apply_mapping(row):
        key = row['identifier'] if row['profession'] == '未知' else row['profession']
        return mapping_dict.get(key, '匹配失败')

    df['职业分类代码'] = df.apply(apply_mapping, axis=1)
    df['职业分类名称'] = df['职业分类代码'].map(code_to_name)

    df.to_excel(output_path, index=False)
    print(f"  已保存: {output_path.name}")
    print(f"  匹配成功: {(df['职业分类代码'] != '匹配失败').sum()}/{len(df)}")

print("分类函数定义完成")

## 执行分类

In [ ]:
# 扫描识别结果目录，处理所有Excel文件
files = list(INPUT_DIR.glob("*.xlsx"))
print(f"发现 {len(files)} 个识别结果文件:\n")
for f in files:
    print(f"  - {f.name}")

print(f"\n{'='*50}")

for file_path in files:
    output_name = f"已分类_一位码_{file_path.name}"
    output_path = OUTPUT_DIR / output_name
    process_file(file_path, output_path)
    time.sleep(0.5)

print(f"\n{'='*50}")
print("所有文件处理完毕！")